In [0]:
# Databricks notebook source
# 1. Khai báo đường dẫn S3 (Sử dụng External Location bạn đã cấu hình)
s3_raw_path = "s3://kien-data-lake-2026/raw"

# 2. Danh sách các file cần nạp
tables = ["olist_orders_dataset", "olist_order_items_dataset", "olist_customers_dataset", "olist_products_dataset"]

# 3. Vòng lặp nạp dữ liệu tự động
for table_name in tables:
    print(f"Đang nạp bảng: {table_name}")
    
    # Đọc CSV
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{s3_raw_path}/{table_name}.csv")
    
    # Ghi vào Delta Table (Bronze Layer)
    # Nếu chưa có database, chạy lệnh: spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

    spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
    
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"bronze.{table_name}")

print("Hoàn thành tầng Bronze!")